In [ ]:
import sys
sys.path.append('../src')
import requests
import xarray as xr
from io import BytesIO
import numpy as np
import glamos_processing as glamos
import meteoswiss_processing as meteo
import pandas as pd

In [ ]:
year = 2018
filename = f'ogd-surface-derived-grid-archive.rhiresm_ch01r.swiss.lv95_{year:04d}0101000000_{year:04d}1201000000.nc'

In [ ]:
r = requests.get('https://data.geo.admin.ch/ch.meteoschweiz.ogd-surface-derived-grid/archive-ch/' + filename)

In [ ]:
with open('data/' + filename, 'wb') as fd:
    for chunk in r.iter_content(chunk_size=128):
        fd.write(chunk)

In [ ]:
ds = xr.open_dataset('data/' + filename, decode_times=False)

In [ ]:
ds

In [ ]:
ds.expand_dims({'year': [year]})

In [ ]:
ds.update({'year':('year', [year])})

In [ ]:
ds.dims

In [ ]:
ds

In [ ]:
ds['TabsM'].values.size

In [ ]:
nu = ds['TabsM'].values.flatten()
np.sum(np.isnan(nu))

In [ ]:
ds.sel(N=1.304e+06, E=2.844e+06, method='nearest')['TabsM'].values

In [ ]:
gl = glamos.get_data()
gl

In [ ]:
temp = ds.sel(N=gl.loc[0].coordy, E=gl.loc[0].coordx, method='nearest')

In [ ]:
ds.time.units.split()[2][0:4]

In [ ]:
def convert_time(ds):
    year = ds.time.units.split()[2][0:4]
    
    dates = pd.date_range(start=f'{year}-01-01', periods=12, freq='MS')

    ds = ds.assign_coords(time=dates)
    return ds

In [ ]:
ds = xr.open_mfdataset('../data/rhiresm/*.nc', preprocess=convert_time, data_vars='all', decode_times=False)

In [ ]:
ds

In [ ]:
p = meteo.extract_precipitation(1982, 2025, skip_dl=True)

In [ ]:
t = meteo.extract_temperature(1982, 2025, skip_dl=True)

In [ ]:
p.merge(t)

In [ ]:
ds = xr.open_mfdataset('../data/tabsm/*.nc', preprocess=convert_time, data_vars='all', decode_times=False)

In [ ]:
ds

In [ ]:
ds.resample(time='QS-OCT').mean()